In [1]:
import dspy

llama3b = dspy.LM('/Llama-3.2-3B-Instruct', temperature=0.7)
gpt4o = dspy.LM('openai/gpt-4o', temperature=0.7)

dspy.configure(lm=gpt4o)

/home/ash/miniconda3/envs/nanites/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import random
from dspy.datasets import DataLoader

kwargs = dict(fields=("claim", "supporting_facts", "hpqa_id", "num_hops"), input_keys=("claim",))
hover = DataLoader().from_huggingface(dataset_name="hover-nlp/hover", split="train", trust_remote_code=True, **kwargs)

hpqa_ids = set()
hover = [
    dspy.Example(claim=x.claim, titles=list(set([y["key"] for y in x.supporting_facts]))).with_inputs("claim")
    for x in hover
    if x["num_hops"] == 3 and x["hpqa_id"] not in hpqa_ids and not hpqa_ids.add(x["hpqa_id"])
]

random.Random(0).shuffle(hover)
trainset, devset, testset = hover[:100], hover[100:200], hover[650:]

Generating test split: 100%|██████████| 4000/4000 [00:00<00:00, 72635.56 examples/s]


In [3]:
example = trainset[0]

print("Claim:", example.claim)
print("Pages that must be retrieved:", example.titles)

Claim: This director is known for his work on Miss Potter. The Academy of Motion Picture Arts and Sciences presents the award in which he was nominated for his work in "Babe".
Pages that must be retrieved: ['Miss Potter', 'Academy Award for Best Director', 'Chris Noonan']


In [4]:
DOCS = {}

def search(query: str, k: int) -> list[str]:
    results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=k)
    results = [x['text'] for x in results]

    for result in results:
        title, text = result.split(" | ", 1)
        DOCS[title] = text

    return results

In [5]:
def search_wikipedia(query: str) -> list[str]:
    """Returns top-5 results and then the titles of the top-5 to top-30 results."""

    topK = search(query, 30)
    titles, topK = [f"`{x.split(' | ')[0]}`" for x in topK[5:30]], topK[:5]
    return topK + [f"Other retrieved pages have titles: {', '.join(titles)}."]

def lookup_wikipedia(title: str) -> str:
    """Returns the text of the Wikipedia page, if it exists."""

    if title in DOCS:
        return DOCS[title]

    results = [x for x in search(title, 10) if x.startswith(title + " | ")]
    if not results:
        return f"No Wikipedia page found for title: {title}"
    return results[0]

In [6]:
instructions = "Find all Wikipedia titles relevant to verifying (or refuting) the claim."
signature = dspy.Signature("claim -> titles: list[str]", instructions)
react = dspy.ReAct(signature, tools=[search_wikipedia, lookup_wikipedia], max_iters=20)

In [7]:
react(claim="David Gregory was born in 1625.").titles[:3]

['David Gregory (physician)']

In [8]:
def top5_recall(example, pred, trace=None):
    gold_titles = example.titles
    recall = sum(x in pred.titles[:5] for x in gold_titles) / len(gold_titles)

    # If we're "bootstrapping" for optimization, return True if and only if the recall is perfect.
    if trace is not None:
        return recall >= 1.0
    
    # If we're just doing inference, just measure the recall.
    return recall

evaluate = dspy.Evaluate(devset=devset, metric=top5_recall, num_threads=16, display_progress=True, display_table=5)

In [9]:
def safe_react(claim: str):
    try:
        return react(claim=claim)
    except Exception as e:
        return dspy.Prediction(titles=[])

evaluate(safe_react)

  0%|          | 0/100 [00:00<?, ?it/s]

Average Metric: 77.33 / 100 (77.3%): 100%|██████████| 100/100 [01:53<00:00,  1.13s/it]

2025/01/14 22:07:00 INFO dspy.evaluate.evaluate: Average Metric: 77.33333333333336 / 100 (77.3%)


,claim,example_titles,trajectory,reasoning,pred_titles,top5_recall
0,The Church of England's movement that inspired the Trinity Episcop...,"[Trinity Episcopal Church (Houghton, Michigan), Oxford Movement, S...","{'thought_0': 'To verify the claim, I need to identify the movemen...",The claim involves identifying the movement within the Church of E...,"[Oxford Movement, Trinity Episcopal Church (Houghton, Michigan), S...",✔️ [1.000]
1,"Red, White & Crüe and this athlete both fight. The french fighter ...","[Bobby Stewart, Red, White &amp; Crüe, Mike Tyson]","{'thought_0': 'The claim suggests a connection between ""Red, White...","The claim connects ""Red, White & Crüe,"" an anthology album by Mötl...","[Red, White & Crüe]",
2,The writer/director/actor from Glen or Glenda and Fernand Rivers s...,"[Glen or Glenda, Ed Wood, Fernand Rivers]","{'thought_0': 'To verify the claim, I need to find information abo...","The claim states that the writer/director/actor from ""Glen or Glen...","[Glen or Glenda, Ed Wood, Fernand Rivers]",✔️ [1.000]
3,The film by Sandi Sissel was released before The End of Suburbia.,"[The End of Suburbia, Sandi Sissel, Chicken Ranch (film)]","{'thought_0': 'To verify the claim, I need to find the release dat...","To verify the claim, I searched for information about Sandi Sissel...","[Chicken Ranch (film), The End of Suburbia]",✔️ [0.667]
4,The actor who played captain hook in the live production with Tayl...,"[Peter Pan Live!, Taylor Louderman, Christopher Walken]","{'thought_0': 'To verify the claim, I need to identify the actor w...",The claim states that the actor who played Captain Hook in a live ...,"[Peter Pan Live!, Christopher Walken, The Deer Hunter]",✔️ [0.667]


77.33

In [14]:
kwargs = dict(teacher_settings=dict(lm=gpt4o), prompt_model=gpt4o, max_errors=999)

tp = dspy.MIPROv2(metric=top5_recall, auto="light", num_threads=16, **kwargs)
optimized_react = tp.compile(react, trainset=trainset, max_bootstrapped_demos=3, max_labeled_demos=0)

2025/01/14 22:15:24 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 3
valset size: 80

2025/01/14 22:43:19 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/01/14 22:43:19 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/01/14 22:43:19 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3


 40%|████      | 8/20 [01:50<02:45, 13.82s/it]


Bootstrapped 3 full traces after 8 examples for up to 1 rounds, amounting to 8 attempts.
Bootstrapping set 3/3


 45%|████▌     | 9/20 [01:35<01:56, 10.62s/it]
2025/01/14 22:46:45 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2025/01/14 22:46:45 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.


...

2025/01/14 22:56:50 ERROR dspy.utils.parallelizer: Error processing item Example({'claim': 'An actor was in the film, Focus, that was directed by Glenn Ficarra. This actor was also in the 2016 film "White Girl".', 'titles': ['White Girl (2016 film)', 'Adrian Martinez (actor)', 'Focus (2015 film)']}) (input_keys={'claim'}): litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-zF4Y24gjYWqHG2pV8aOzwU5v on tokens per min (TPM): Limit 450000, Used 446084, Requested 8172. Please try again in 567ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}. Set `provide_traceback=True` to see the stack trace.


Average Metric: 15.33 / 21 (73.0%):  92%|█████████▎| 74/80 [01:04<00:09,  1.62s/it]

2025/01/14 22:56:54 ERROR dspy.utils.parallelizer: Error processing item Example({'claim': 'Archers of Loaf is an indie band. The band who had a song called Shelf in the Room is not.', 'titles': ['Shelf in the Room', 'Archers of Loaf', 'Days of the New']}) (input_keys={'claim'}): litellm.RateLimitError: RateLimitError: OpenAIException - Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-zF4Y24gjYWqHG2pV8aOzwU5v on tokens per min (TPM): Limit 450000, Used 446636, Requested 8332. Please try again in 662ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}. Set `provide_traceback=True` to see the stack trace.


Average Metric: 19.00 / 26 (73.1%): 100%|██████████| 80/80 [01:18<00:00,  1.02it/s]

2025/01/14 22:57:05 INFO dspy.evaluate.evaluate: Average Metric: 18.999999999999996 / 80 (23.7%)
2025/01/14 22:57:05 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [72.08, 23.75]
2025/01/14 22:57:05 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 72.08
2025/01/14 22:57:05 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/01/14 22:57:05 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/01/14 22:57:05 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 72.08!


In [15]:
optimized_react(claim="The author of the 1960s unproduced script written for The Beatles, Up Against It, and Bernard-Marie Koltès are both playwrights.").titles

['Up Against It', 'Joe Orton', 'Bernard-Marie Koltès']

In [16]:
dspy.inspect_history(n=2)





[2025-01-14T23:00:21.260838]

System message:

Your input fields are:
1. `claim` (str)
2. `trajectory` (str)

Your output fields are:
1. `next_thought` (str)
2. `next_tool_name` (Literal[search_wikipedia, lookup_wikipedia, finish])
3. `next_tool_args` (dict[str, Any])

All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## claim ## ]]
{claim}

[[ ## trajectory ## ]]
{trajectory}

[[ ## next_thought ## ]]
{next_thought}

[[ ## next_tool_name ## ]]
{next_tool_name}        # note: the value you produce must be one of: search_wikipedia; lookup_wikipedia; finish

[[ ## next_tool_args ## ]]
{next_tool_args}        # note: the value you produce must be pareseable according to the following JSON schema: {"type": "object"}

[[ ## completed ## ]]

In adhering to this structure, your objective is: 
        Find all Wikipedia titles relevant to verifying (or refuting) the claim.
        
        You will be given `claim` and your goal is to fini

In [17]:
optimized_react.save("optimized_react.json")

loaded_react = dspy.ReAct("claim -> titles: list[str]", tools=[search_wikipedia, lookup_wikipedia], max_iters=20)
loaded_react.load("optimized_react.json")

loaded_react(claim="The author of the 1960s unproduced script written for The Beatles, Up Against It, and Bernard-Marie Koltès are both playwrights.").titles

['Up Against It', 'Joe Orton', 'Bernard-Marie Koltès']